# Gene Signature Scoring of Epithelial and Malignant States

Scores published gene signatures (KAC / NF-κB / interferon, Peng et al. 2025;
SCLC subtype, NE, tuft, ionocyte and hillock programmes, Ireland et al. 2025;
Tcell inflamed GEP, CIN25, YAP, Maynard RD/PD) on the epithelial/tumour object,
then compares programme activity across cell states, RECIST treatment stages and
donors, and profiles neuroendocrine, antigen-presentation and ADC-target markers
in the malignant compartment.

In [ ]:
# !pip install --quiet scanpy statsmodels
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/tumour/pulmonary_epi_Lin/

In [ ]:
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu, zscore
from statsmodels.stats.multitest import multipletests

## Config

In [ ]:
input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join.h5ad"

output_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join_scores.h5ad"
figures_output_dir = "./figures"
exploratory_output_dir = "./figures/exploratory_scores"
tables_output_dir = "./tables"

state_col = "final_state"
base_label_col = "cell_type_new"
malignant_label = "Malignant"
unassigned_label = "unassigned"
recist_col = "recist_v3"
donor_col = "ID_2"
counts_layer = "raw_counts"
embedding_basis = "X_draw_graph_fa"

# --- signature scoring (sc.tl.score_genes) ---
# CSV files: one column per signature, gene symbols down the rows
signature_files = ["Peng_et_al_2025.csv", "Ireland_et_al_2025_Nature.csv"]
inline_signatures = {
    "GEP_score": ["HLA-DQA1", "PSMB10", "HLA-DRB1", "HLA-E", "CD276", "CD38", "CXCL9", "LAG3", "IDO1", "STAT1",
                  "ENTPD1", "PDCD1LG2", "CD274", "CMKLR1", "CXCR6", "CCL5", "CD8A", "NKG7", "TIGIT", "CD27", "CTLA4"],
    "CIN25_score": ["PRC1", "CCT5", "H2AFZ", "PCNA", "TGIF2", "MAD2L1", "RFC4", "NCAPD2", "FEN1", "MCM2", "MCM7",
                    "RNASEH2A", "TRIP13", "CCNB2", "CDK1", "RAD51AP1", "UBE2C", "CDC45", "TOP2A", "TPX2", "FOXM1",
                    "MELK", "TTK", "KIF20A", "ESPL1"],
    "YAP_score": ["GAS6", "EPAS1", "BMP1", "PHLDB2", "TNNC1", "CD55", "GLS", "CLIC3", "CLIC5", "CITED2", "CYBRD1",
                  "EPB41L5", "C1orf198", "DLC1", "BEX2", "AGER", "ITPR2", "NEDD9", "PTPRE", "MAP1B", "CAVIN1", "CCN3",
                  "YAP1", "TEAD1", "TEAD2", "TEAD3", "TEAD4"],
    "RD_maynard_score": ["LAMP1", "AGER", "FOXA2", "NAPSA", "HOPX", "IGFBP2", "SFTA3", "NKX2-1", "GATA6", "ABCA3",
                         "SCGB3A1", "AQP4", "CLDN18", "PGC", "SFTPD", "SFTPB", "SFTPC"],
    "PD_maynard_score": ["SERPINE1", "ANXA2", "PLAT", "PLAUR", "PLAU", "IDO1", "KYNU", "QPRT"],
}
min_signature_genes = 3
score_n_bins = 50
score_random_state = 0

# --- states, stages, donors ---
normal_state_order = [
    "Alveolar cell type I", "Alveolar cell type II", "Transitional",
    "Club cell", "Non-malignant epithelial", "Goblet cell",
]
malignant_state_order = [
    "1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
    "4_Inflammatory EMT", "6_Proliferative", "5_Neurod_EMT-like",
]
state_order = normal_state_order + malignant_state_order
state_colors = {
    "Malignant": "#023fa5",
    "Non-malignant epithelial": "#84bcf0",
    "Transitional": "#63ffac",
    "Alveolar cell type I": "#7d87b9",
    "Alveolar cell type II": "#bec1d4",
    "Club cell": "#4a6fe3",
    "Goblet cell": "#8e063b",
    "1_Epithelial baseline": "#8dd3c7",
    "2_Differentiated epithelial": "#bebada",
    "3_Devp_stem-like": "#80b1d3",
    "4_Inflammatory EMT": "#fccde5",
    "5_Neurod_EMT-like": "#bc80bd",
    "6_Proliferative": "#ffed6f",
    "unassigned": "grey",
}
recist_order = ["0_adj_normal", "1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
treatment_order = ["1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
tumour_donor_order = [
    "TN007", "TN008", "TN004", "TN005", "TN006", "TN002", "TN001", "TN009", "TN003",
    "P012", "P006", "P001", "P011", "P003", "P002", "P008", "P005", "P013", "P009",
    "R001", "R010", "R009", "R006", "R003", "R007", "R005",
]

# --- programme heatmaps (mean score per group, z-scored across groups) ---
heatmap_scores = [
    "MYC_Targets_score", "SCLC-A_score", "SCLC-A2_score", "SCLC-N_score", "SCLC-P_score",
    "YAP_score", "KAC-Human_score", "Interferon-Human_score", "CIN25_score",
    "NF-kB-Human_score", "RD_maynard_score", "PD_maynard_score",
]
heatmap_mark_z = 0.25          # cells with z > this get "*"; None to disable (not a significance test)
heatmap_cmap = "vanimo"

# --- KAC treatment-stage comparison within malignant states ---
stats_scores = ["KAC-Human_score"]
stats_unit = "patient"         # "patient" (median per patient) or "cell"
min_cells_per_patient = 20
correction_method = "fdr_bh"   # post-hoc tests, corrected within each state
alpha = 0.05

# --- marker panels ---
mhc_panels = {                 # Figure S4B
    "mhc_genes": ["HLA-A", "HLA-B", "HLA-C"],
    "mhc_genes_2": ["B2M", "TAP1", "TAP2"],
    "mhc_genes_3": ["CALR", "PSMB8", "PSMB9", "NLRC5"],
}
ne_panels_malignant = {
    "mhc_genes": ["HLA-A", "HLA-B", "HLA-C"],
    "mhc_genes_2": ["B2M", "TAP1", "TAP2", "CALR", "PSMB8", "PSMB9", "XBP1", "NLRC5", "CD274", "DDIT3", "ATF4",
                    "STAT1", "IRF1", "IRF9", "IFIT1", "IFIT2", "IFIT3", "ISG15", "MX1", "MX2", "OAS1", "OAS3",
                    "IFNG", "TNF", "IL2", "CD47"],
    "canonical-1": ["ASCL1", "POU2F3", "NEUROD1", "INSM1", "DLL3", "NCAM1", "HEY1"],
    "secretory": ["SYP", "CHGA", "CHGB", "CGA", "SCG2"],
    "intermediate": ["EZH2", "YAP1", "POU6F2"],
    "Neuronal-like": ["NGF", "SYN3", "SYT1", "SYT6", "SYT12", "SYT14", "NLGN1", "GRIA4", "SLC6A3", "CNTNAP2",
                      "CDH11", "IL33", "MAPT", "SEPTIN7"],
    "Basal": ["EPCAM", "KRT8", "KRT18", "KRT19", "KRT14", "KRT5", "SOX2", "KRT17", "SOX4"],
    "Neuron": ["RBFOX3", "SNAP25", "TUBB3", "S100B"],
}
adc_panels = {
    "1": ["TACSTD2", "ERBB2", "EGFR", "MET", "ERBB3"],
    "2": ["CD276", "NECTIN4", "PTK2", "SLC39A6"],
    "3": ["MSLN", "DLL3", "CD47", "SLC3A2", "ITGA2", "ITGA3", "EPHA2", "TFRC"],
}
neural_emt_state = "5_Neurod_EMT-like"
neural_emt_panels = { 
    "canonical-1": ["ASCL1", "POU2F3", "NEUROD1", "INSM1", "DLL3", "NCAM1", "HEY1"],
    "secretory": ["SYP", "CHGA", "CHGB", "CGA", "SCG2"],
    "Neuronal-like": ["SYN3", "SYT1", "SYT12", "NRCAM", "NXPH1", "EGLN3", "NLGN1", "GRIA4", "SLC6A3", "CNTNAP2",
                      "POU6F2", "SOX4", "MAPT", "SEPTIN7", "TP63"],
    "Neuron": ["RBFOX3", "SNAP25", "TUBB3", "S100B"],
}
yap_tead_panel = {"yapfam": ["YAP1", "ARHGEF10", "TEAD1", "SNX7"]}

for d in (figures_output_dir, exploratory_output_dir, tables_output_dir):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = figures_output_dir

## 1. Load and normalise

`X` is reset to raw counts before normalisation. Gene symbols are upper-cased so
signature lists match regardless of source casing.

In [ ]:
adata = sc.read(input_path)
adata.X = adata.layers[counts_layer].copy() if counts_layer in adata.layers else adata.X.copy()
adata.layers[counts_layer] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.var_names = adata.var_names.str.upper()
if not adata.var_names.is_unique:
    print(f"{adata.var_names.duplicated().sum()} duplicate symbols after upper-casing; made unique")
    adata.var_names_make_unique()

missing_colors = set(adata.obs[state_col].astype(str).unique()) - set(state_colors)
assert not missing_colors, f"states without a colour: {missing_colors}"
adata

## 2. Score gene signatures

All signatures use the same parameters: genes absent from the dataset are dropped,
the control set size equals the number of genes present, and signatures with fewer
than `min_signature_genes` present are skipped. Scores are computed on the full
epithelial object, so the control-gene background includes non-malignant cells.

In [ ]:
def load_signature_csv(path):
    """One signature per column; blank cells and 'Unnamed' columns ignored."""
    df = pd.read_csv(path, dtype=str)
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    return {f"{col}_score": df[col].dropna().str.strip().str.upper().tolist() for col in df.columns}

signatures = {}
for path in signature_files:
    signatures.update(load_signature_csv(path))
signatures.update({k: [g.upper() for g in v] for k, v in inline_signatures.items()})

coverage = []
for score_name, genes in signatures.items():
    present = [g for g in dict.fromkeys(genes) if g in adata.var_names]
    missing = sorted(set(genes) - set(present))
    coverage.append({"score": score_name, "n_genes": len(set(genes)), "n_present": len(present),
                     "missing": ", ".join(missing)})
    if len(present) < min_signature_genes:
        continue
    sc.tl.score_genes(adata, gene_list=present, ctrl_size=len(present), n_bins=score_n_bins,
                      score_name=score_name, use_raw=False, random_state=score_random_state)

coverage_df = pd.DataFrame(coverage)
coverage_df.to_csv(f"{tables_output_dir}/signature_gene_coverage.csv", index=False)
coverage_df

## 3. Save scored object

In [ ]:
adata_out = adata.copy()
adata_out.X = adata_out.layers[counts_layer].copy()
adata_out.uns.pop("log1p", None)
adata_out.write(output_path, compression="gzip")
del adata_out

## 4. Programme activity across states and treatment stages

In [ ]:
def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s).strip("_")

def plot_state_violin(obs_df, y, states, stages=None, colors=state_colors, horizontal=False,
                      show_n=False, figsize=None, save_path=None):
    """Violin of `y` per state, optionally split by RECIST stage; coloured by state."""
    df = obs_df[[state_col, recist_col, y]].copy()
    df[state_col], df[recist_col] = df[state_col].astype(str), df[recist_col].astype(str)
    df = df[df[state_col].isin(states)]
    if stages is None:
        df["group"] = df[state_col]
        order = [s for s in states if s in set(df["group"])]
        palette = {g: colors[g] for g in order}
    else:
        df = df[df[recist_col].isin(stages)]
        df["group"] = df[state_col] + " | " + df[recist_col]
        present = set(df["group"])
        order = [f"{s} | {t}" for s in states for t in stages if f"{s} | {t}" in present]
        palette = {g: colors[g.split(" | ")[0]] for g in order}

    long_side = max(4.5, 0.35 * len(order))
    fig, ax = plt.subplots(figsize=figsize or ((4, long_side) if horizontal else (long_side, 4.5)))
    xy = dict(y="group", x=y) if horizontal else dict(x="group", y=y)
    sns.violinplot(data=df, **xy, hue="group", order=order, hue_order=order, palette=palette,
                   legend=False, cut=0, linewidth=0.5, density_norm="width", ax=ax)
    label = y.replace("_score", "").replace("_", " ")
    if horizontal:
        ax.set(xlabel=label, ylabel="")
    else:
        ax.set(xlabel="", ylabel=label)
        ax.tick_params(axis="x", rotation=90)
    if show_n and not horizontal:
        counts = df["group"].value_counts()
        lo, hi = df[y].min(), df[y].max()
        for i, g in enumerate(order):
            ax.text(i, hi + 0.05 * (hi - lo), f"n={counts[g]}", ha="center", va="bottom", fontsize=8, rotation=90)
        ax.set_ylim(lo, hi + 0.18 * (hi - lo))
    sns.despine()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
score_cols = [c for c in signatures if c in adata.obs.columns]
for score in score_cols:
    plot_state_violin(adata.obs, score, state_order, stages=recist_order,
                      save_path=f"{exploratory_output_dir}/{safe_name(score)}_violin_state_recist.pdf")

### KAC-Human score

In [ ]:
score = "KAC-Human_score"
vmin, vmax = adata.obs[score].min(), adata.obs[score].max()

fig, axes = plt.subplots(1, len(recist_order), figsize=(15, 4))
for ax, stage in zip(axes, recist_order):
    sc.pl.embedding(adata[adata.obs[recist_col] == stage], basis=embedding_basis, color=score, size=30,
                    cmap="CMRmap", vmin=vmin, vmax=vmax, ax=ax, show=False, title=stage)
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/25_KAC_Human_score_drawgraph_split_recist.pdf", dpi=150, bbox_inches="tight")
plt.show()

adata_mal = adata[(adata.obs[base_label_col] == malignant_label)
                  & (adata.obs[state_col] != unassigned_label)].copy()

plot_state_violin(adata_mal.obs, score, malignant_state_order, show_n=True,
                  save_path=f"{figures_output_dir}/26_KAC_Human_score_violin_malignant_state.pdf")
plot_state_violin(adata_mal.obs, score, malignant_state_order, stages=treatment_order, horizontal=True,
                  figsize=(2.5, 0.3 * len(malignant_state_order) * len(treatment_order)),
                  save_path=f"{figures_output_dir}/27_KAC_Human_score_violin_malignant_state_recist.pdf")

### Treatment-stage comparison within malignant states

Kruskal–Wallis across stages per state, pairwise Mann–Whitney only where the
omnibus test is significant, corrected within state. `stats_unit = "patient"`
tests one median per patient; `"cell"` reproduces the original cell-level analysis.

In [ ]:
def stage_comparison(obs_df, value):
    df = obs_df[[state_col, recist_col, donor_col, value]].copy()
    for c in (state_col, recist_col, donor_col):
        df[c] = df[c].astype(str)
    df = df[df[state_col].isin(malignant_state_order) & df[recist_col].isin(treatment_order)]
    if stats_unit == "patient":
        grouped = df.groupby([state_col, recist_col, donor_col])[value]
        df = pd.concat([grouped.median(), grouped.size().rename("n_cells")], axis=1).reset_index()
        df = df[df["n_cells"] >= min_cells_per_patient]

    rows = []
    for state in malignant_state_order:
        d = df[df[state_col] == state]
        groups = {t: d.loc[d[recist_col] == t, value].values for t in treatment_order}
        groups = {t: v for t, v in groups.items() if len(v) >= 2}
        n = {t: len(v) for t, v in groups.items()}
        if len(groups) < 2:
            rows.append({"state": state, "test": "skipped", "n": str(n)})
            continue
        h, p_omni = kruskal(*groups.values())
        rows.append({"state": state, "test": "kruskal", "comparison": "all", "statistic": h, "p": p_omni, "n": str(n)})
        if p_omni >= alpha:
            continue
        stages = list(groups)
        posthoc = []
        for i in range(len(stages)):
            for j in range(i + 1, len(stages)):
                a, b = stages[i], stages[j]
                u, p = mannwhitneyu(groups[a], groups[b], alternative="two-sided")
                posthoc.append({"state": state, "test": "mannwhitneyu", "comparison": f"{a} vs {b}",
                                "statistic": u, "p": p, "n": f"{n[a]} vs {n[b]}",
                                "median_a": np.median(groups[a]), "median_b": np.median(groups[b])})
        for r, pa in zip(posthoc, multipletests([r["p"] for r in posthoc], method=correction_method)[1]):
            r["p_adj"] = pa
        rows.extend(posthoc)
    out = pd.DataFrame(rows)
    out.insert(0, "score", value)
    out.insert(1, "unit", stats_unit)
    return out

stats_df = pd.concat([stage_comparison(adata_mal.obs, s) for s in stats_scores], ignore_index=True)
stats_df.to_csv(f"{tables_output_dir}/signature_treatment_stage_stats_{stats_unit}.csv", index=False)
stats_df

## 5. Programme heatmaps (Figure 2)

Mean score per group, z-scored across groups for each programme. `*` marks
z > `heatmap_mark_z`; it is a display threshold, not a statistical test.

In [ ]:
def programme_heatmap(obs_df, group_col, scores, group_order=None, cluster_cols=True,
                      figsize=(3.5, 5), save_path=None):
    scores = [s for s in scores if s in obs_df.columns]
    mat = obs_df.groupby(group_col, observed=True)[scores].mean()
    if group_order is not None:
        mat = mat.loc[[g for g in group_order if g in mat.index]]
    plot_mat = mat.apply(zscore, axis=0).T
    plot_mat.index = plot_mat.index.str.replace("_score", "", regex=False).str.replace("_", " ", regex=False)
    annot = (plot_mat > heatmap_mark_z).replace({True: "*", False: ""}) if heatmap_mark_z is not None else None

    g = sns.clustermap(plot_mat, cmap=heatmap_cmap, center=0, linewidths=0.05, linecolor="lightgray",
                       figsize=figsize, metric="correlation", method="average",
                       row_cluster=True, col_cluster=cluster_cols,
                       annot=annot, fmt="", annot_kws={"fontsize": 9, "fontweight": "bold"},
                       cbar_kws={"label": "Z-score"})
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)
    if save_path:
        g.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return plot_mat

state_z = programme_heatmap(adata_mal.obs, state_col, heatmap_scores, 
                            save_path=f"{figures_output_dir}/28a_final_state_clustered_heatmap_scores.pdf") #Figure 2
donor_z = programme_heatmap(adata_mal.obs, donor_col, heatmap_scores, group_order=tumour_donor_order,
                            cluster_cols=False, figsize=(9, 5),
                            save_path=f"{figures_output_dir}/28b_donor_heatmap_rowcluster_only.pdf") ##Supplementary Fig. 4

## 6. Marker panels in malignant states by treatment stage

Adjacent-normal samples are excluded. Dot colour is scaled per gene across groups.

In [ ]:
adata_tum = adata_mal[adata_mal.obs[recist_col].isin(treatment_order)].copy()
adata_tum.obs["state_recist"] = pd.Categorical(
    adata_tum.obs[state_col].astype(str) + " | " + adata_tum.obs[recist_col].astype(str),
    categories=[f"{s} | {t}" for s in malignant_state_order for t in treatment_order
                if ((adata_tum.obs[state_col] == s) & (adata_tum.obs[recist_col] == t)).any()],
)

def marker_dotplot(ad, panels, groupby, figsize, save_path=None, plot_fn=sc.pl.dotplot, **kwargs):
    present = {k: [g for g in v if g in ad.var_names] for k, v in panels.items()}
    missing = {k: sorted(set(v) - set(present[k])) for k, v in panels.items()}
    if any(missing.values()):
        print("not found:", {k: v for k, v in missing.items() if v})
    present = {k: v for k, v in present.items() if v}
    axes_dict = plot_fn(ad, present, groupby=groupby, use_raw=False, standard_scale="var", cmap="Blues",
                        figsize=figsize, show=False, swap_axes=True, dendrogram=False, **kwargs)
    for ax in axes_dict.values():
        for label in ax.get_yticklabels():
            label.set_style("italic")
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

marker_dotplot(adata_tum, mhc_panels, "state_recist", figsize=(6.3, 2.5),
               save_path=f"{figures_output_dir}/30_dotplot_markers_MHC_antigen_presentation_recist.pdf") #Supplementary Fig. 4
marker_dotplot(adata_tum, ne_panels_malignant, "state_recist", figsize=(10, 15), dot_min=0.1,
               save_path=f"{exploratory_output_dir}/dotplot_NE_immune_markers_state_recist.pdf")
marker_dotplot(adata_tum, adc_panels, "state_recist", figsize=(6.5, 4.8),
               save_path=f"{exploratory_output_dir}/dotplot_ADC_targets_state_recist.pdf") #Figure 2

## 7. Neural EMT-like state (state 5)

Neuroendocrine/neuronal markers and YAP–TEAD expression within state 5 across
treatment stages and donors.

In [ ]:
adata_ne = adata_tum[adata_tum.obs[state_col] == neural_emt_state].copy()
adata_ne.obs["state_recist"] = adata_ne.obs["state_recist"].cat.remove_unused_categories()
print(adata_ne.obs[recist_col].value_counts())

marker_dotplot(adata_ne, neural_emt_panels, "state_recist", figsize=(3.5, 7.5), dot_min=0.05,
               save_path=f"{figures_output_dir}/31a_dotplot_markers_NE_like_state5_recist.pdf")

adata_ne.obs[donor_col] = pd.Categorical(
    adata_ne.obs[donor_col].astype(str),
    categories=[d for d in tumour_donor_order if d in set(adata_ne.obs[donor_col].astype(str))])
marker_dotplot(adata_ne, yap_tead_panel, donor_col, figsize=(8, 1), plot_fn=sc.pl.matrixplot,
               save_path=f"{figures_output_dir}/31b_matrixplot_YAP_TEAD_state5_donor.pdf")

### Exploratory: state-5 DE across treatment stages (not in figures)

In [ ]:
sc.tl.rank_genes_groups(adata_ne, groupby="state_recist", method="wilcoxon")
sc.pl.rank_genes_groups(adata_ne, n_genes=30, sharey=False)